# 🚀 DevSync AI - LoRA Fine-Tuning

This notebook fine-tunes a small LLM on DevSync-specific Q&A data using LoRA (Low-Rank Adaptation).

**Requirements:**
- Google Colab with GPU (free tier T4 works!)
- ~15-30 minutes training time

**Output:**
- LoRA adapter files to use with Ollama

In [ ]:
# Install required packages
!pip install -q transformers datasets peft accelerate bitsandbytes trl
!pip install -q torch --upgrade

In [ ]:
# Check GPU
import torch
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Training data - DevSync Q&A pairs
TRAINING_DATA = [
  {"instruction": "How do I add a profile picture?", "output": "Go to Dashboard → Profile, click your avatar, and upload an image. Use a professional headshot with good lighting, minimum 400x400px."},
  {"instruction": "How do I update my bio?", "output": "Go to Dashboard → Profile → Edit Bio. Keep it 150-300 characters. Format: Who you are + What you do + What makes you unique."},
  {"instruction": "How do I add skills?", "output": "Go to Dashboard → Skills → Add Skill. Enter skill name, category, and proficiency (0-100%). Aim for 8-15 skills with honest levels."},
  {"instruction": "How do I add a project?", "output": "Go to Dashboard → Projects → Add Project. Include title, description, technologies, GitHub link, and live demo URL. Showcase 3-6 quality projects."},
  {"instruction": "How do I add work experience?", "output": "Go to Dashboard → Experience → Add Experience. Enter company, role, dates, and achievements. Focus on metrics like 'Reduced load time by 40%'."},
  {"instruction": "How do I add education?", "output": "Go to Dashboard → Education → Add Education. Enter institution, degree, field of study, and graduation date. Include relevant coursework."},
  {"instruction": "How do I add certifications?", "output": "Go to Dashboard → Certifications → Add Certification. Enter name, issuer, dates, and credential URL. Popular: AWS, Google Cloud, Meta certifications."},
  {"instruction": "How can I improve my portfolio?", "output": "Check your Portfolio Analysis score. Key areas: 1) Professional photo, 2) Compelling bio, 3) 3-6 projects with demos, 4) 8-15 skills, 5) Complete experience section."},
  {"instruction": "What should I write in my title?", "output": "Be specific like 'Full Stack Developer | React & Node.js' rather than generic 'Developer'. Include your main technologies."},
  {"instruction": "How many skills should I add?", "output": "Aim for 8-15 skills. Include both technical (React, Python) and soft skills (Communication). Be honest with proficiency levels."},
  {"instruction": "How do I write a good project description?", "output": "Start with what it does and why you built it. Highlight the problem solved, key features, and technologies. Include metrics if possible."},
  {"instruction": "What do recruiters look for?", "output": "Clear professional identity, real projects with code links, problem-solving in descriptions, continuous learning (certifications), and good communication."},
  {"instruction": "Should I use GitHub Copilot?", "output": "AI tools like Copilot boost productivity 30-50% for coding tasks. Add AI tools to your Skills section to show you're up-to-date with modern practices."},
  {"instruction": "What technologies should I learn?", "output": "Focus on your career goals. For web dev: React/Next.js, Node.js, PostgreSQL, Docker. Learn one thing deeply before broadening."},
  {"instruction": "How do I prepare for interviews?", "output": "Use your DevSync portfolio as talking points. Practice explaining each project: the problem, your approach, challenges, and results. Check Portfolio Analysis for gaps."},
  {"instruction": "How do I link my GitHub?", "output": "Go to Dashboard → Profile → Social Links → Add GitHub URL. For projects, add repository links individually. Keep repos with good READMEs."},
  {"instruction": "What makes a good profile photo?", "output": "Professional headshot with good lighting, neutral background, and a friendly smile. Minimum 400x400px. Profiles with photos get 14x more views."},
  {"instruction": "How do I get a higher portfolio score?", "output": "Complete all sections: photo, bio, title, 8+ skills, 3+ projects with demos, experience, and education. Each section adds to your score."},
  {"instruction": "What is DevSync?", "output": "DevSync is a developer portfolio platform with AI assistance. Create your profile, showcase projects, list skills, and get personalized improvement suggestions."},
  {"instruction": "How does the AI assistant work?", "output": "DevSync AI uses a locally-hosted language model with RAG for accurate, context-aware responses. Your data stays private - no external API calls."},
  {"instruction": "What tech stack does DevSync use?", "output": "Django REST backend, Next.js frontend with TypeScript and Tailwind, Ollama for AI, ChromaDB for RAG, PostgreSQL database, Docker deployment."},
  {"instruction": "Hello", "output": "Hi! I'm DevSync AI. I can help you build and improve your developer portfolio. Ask me about adding projects, skills, or how to improve your profile."},
  {"instruction": "Thank you", "output": "You're welcome! Good luck with your portfolio. Remember to check your Portfolio Analysis for personalized improvement tips."},
  {"instruction": "What can you help with?", "output": "I help with DevSync portfolio features: Profile setup, Skills management, Project showcasing, Experience/Education sections, and portfolio improvement tips."},
]

In [ ]:
# Format data for training
from datasets import Dataset

def format_prompt(example):
    return f"""### Instruction:
{example['instruction']}

### Response:
{example['output']}"""

# Create dataset
formatted_data = [{"text": format_prompt(d)} for d in TRAINING_DATA]
dataset = Dataset.from_list(formatted_data)
print(f"Dataset size: {len(dataset)}")
print(f"\nExample:\n{dataset[0]['text']}")

In [ ]:
# Load base model (TinyLlama - small and fast)
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

# Use TinyLlama - 1.1B params, fits in Colab free tier
MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# 4-bit quantization for memory efficiency
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Load model
print(f"Loading {MODEL_ID}...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Model loaded!")

In [ ]:
# Configure LoRA
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Prepare model for training
model = prepare_model_for_kbit_training(model)

# LoRA configuration - small rank for speed
lora_config = LoraConfig(
    r=8,  # Low rank for faster training
    lora_alpha=16,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

# Apply LoRA
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Training configuration
from transformers import TrainingArguments
from trl import SFTTrainer

# Training arguments - optimized for Colab free tier
training_args = TrainingArguments(
    output_dir="./devsync-lora",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    weight_decay=0.01,
    logging_steps=10,
    save_strategy="epoch",
    fp16=True,
    optim="paged_adamw_8bit",
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    report_to="none",  # Disable wandb
)

# Create trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    dataset_text_field="text",
    tokenizer=tokenizer,
    args=training_args,
    max_seq_length=256,
    packing=True,
)

print("Trainer ready!")

In [ ]:
# 🚀 Start training!
print("Starting fine-tuning... This takes ~15-30 minutes on Colab free tier.")
trainer.train()
print("\n✅ Training complete!")

In [ ]:
# Save the LoRA adapter
ADAPTER_PATH = "./devsync-lora-adapter"
model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)
print(f"LoRA adapter saved to {ADAPTER_PATH}")

In [ ]:
# Test the fine-tuned model
from transformers import pipeline

# Create pipeline
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=100,
    temperature=0.7,
)

# Test questions
test_questions = [
    "How do I add a project?",
    "What is DevSync?",
    "Should I use GitHub Copilot?",
]

for q in test_questions:
    prompt = f"### Instruction:\n{q}\n\n### Response:\n"
    result = pipe(prompt)[0]['generated_text']
    response = result.split("### Response:\n")[-1].strip()
    print(f"Q: {q}")
    print(f"A: {response}\n")

In [ ]:
# Download the adapter files
from google.colab import files
import shutil

# Zip the adapter
shutil.make_archive("devsync-lora-adapter", "zip", ADAPTER_PATH)

# Download
files.download("devsync-lora-adapter.zip")
print("\n📦 Download complete! Extract and use with Ollama.")

## 📋 Next Steps: Use with Ollama

1. **Download** the `devsync-lora-adapter.zip` file

2. **Create Modelfile** for Ollama:
```
FROM tinyllama
ADAPTER ./devsync-lora-adapter
PARAMETER temperature 0.7
PARAMETER num_predict 100
SYSTEM "You are DevSync AI, a helpful assistant for the DevSync developer portfolio platform."
```

3. **Build the model**:
```bash
ollama create devsync-ai -f Modelfile
```

4. **Test it**:
```bash
ollama run devsync-ai "How do I add a project?"
```

5. **Update DevSync backend** to use `devsync-ai` model:
```bash
export OLLAMA_MODEL=devsync-ai
```